# 04 — Disease-relevant regulons

Finds regulons that are relevant to the disease: survival-associated (Cox) **or** hallmark-enriched (GO + Lin similarity), **and** co-expression-validated in an independent cohort.
`{prefix}` = `CANCER.lower()` (e.g. `brca`), `{val}` = `VAL_NAME.lower()` (e.g. `metabric`); paths are set in the config cell.

## Inputs

| File | From | Format |
|---|---|---|
| `{prefix}_output/eigengenes.csv` | 02 | regulons × patients; index = regulon ID (int), columns = TCGA patient IDs; values = PC1 activity score |
| `{prefix}_output/clinical_data.csv` | 03 | index = `PATIENT_ID`; columns `OS_STATUS` (0/1), `OS_MONTHS` (float) |
| `{prefix}_output/regulons.json` | 02 | `{regulon ID (str): [member genes, Ensembl IDs]}` |
| `{prefix}_zscored.csv` (in `base_dir`) | 01 | genes (symbols) × patients, z-scored; only the gene list is used (GO background) |
| `GO_DIR/go-basic.obo` | GO website | GO term hierarchy (DAG) |
| `GO_DIR/goa_human.gaf.gz` | GO website (downloaded if missing) | human gene → GO term annotations |
| `miner/data/identifier_mappings.txt` | miner package | Ensembl ↔ gene symbol mapping |
| `val_dir/data_mrna*.txt` | validation cohort (cBioPortal) | rows = genes (`Hugo_Symbol`, `Entrez_Gene_Id`), columns = samples; log2 expression |
| `val_dir/data_clinical_patient.txt` | validation cohort (cBioPortal) | `#` header lines, then `PATIENT_ID`, `OS_MONTHS`, `OS_STATUS` (`"1:DECEASED"`), optional `VITAL_STATUS` |

## Outputs (all in `{prefix}_output/` unless noted)

| File | Format |
|---|---|
| `regulon_cox_results.csv` | index = regulon ID; `HR`, `p_value`, `disease_relevant` (True if p ≤ 0.05) |
| `disease_relevant_regulons_survival_only.csv` | no header; one regulon ID per line (Cox p ≤ 0.05) |
| `GO_DIR/goa_human.gaf` | unzipped GO annotation file (written in the GO_DIR folder) |
| `go_bio_symbol.pkl` (+ `_checkpoint`) | dict `{regulon ID (str): [significant GO BP term IDs]}` (FDR ≤ 0.05); cached, ~47 min to compute |
| `lin_cache.pkl` (+ `_checkpoint`) | dict `{regulon GO term: {hallmark GO term: Lin similarity 0–1}}`; cached, ~15 min to compute |
| `disease_relevant_regulons_survival_or_hallmark.csv` | no header; regulon IDs that are survival-significant OR hallmark-enriched |
| `{val}_zscored.pkl` | pickled DataFrame, genes (symbols) × validation samples, z-scored per gene; used by 05 |
| `{val}_clinical.csv` | index = `PATIENT_ID`; `OS_MONTHS`, `OS_STATUS` (0/1), `DSS_STATUS` (0/1/NaN, only if the cohort records cause of death); used by 05 |
| `{val}_random_regulon_variance.csv` | null distribution; index = gene-set of size N; columns `N`, `Variance` (variance of all regulon genes averaged per patient), `Variance_explained` (PC1); 250 rows per size; cached |
| `{val}_regulon_coherence.csv` | one row per testable regulon: `N`, `Variance`, `Variance_expained`, `Modules` (row position), `Regulon` (ID), `Perm_pass` (0/1), `PC1_pass` (0/1) |
| `{val}_coexp_validated_regulons.csv` | no header; regulon IDs passing permutation p ≤ 0.05 AND PC1 ≥ 0.3 |
| `final_disease_relevant_regulons.csv` | no header; **final set** = (survival OR hallmark) AND co-expression validated. Written twice: first as the union (cell 20), then overwritten by the final set (last cell). Always read with `header=None` |

**BRCA reference numbers:** 1,219 survival-significant · 818 hallmark-enriched · 1,919 union · 3,134 METABRIC-validated · **869 final**.

In [1]:
# Necessary imports and path specifications
import pandas as pd
import numpy as np
import os
import pickle
import warnings
import gzip
import json
import requests
import time
import miner.miner as miner
import glob, io, contextlib

from miner.miner import *
from collections import defaultdict, Counter
from goatools.anno.gaf_reader import GafReader
from lifelines import CoxPHFitter
from goatools.obo_parser import GODag
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from goatools.go_enrichment import GOEnrichmentStudy
from goatools.semantic import lin_sim, TermCounts

warnings.filterwarnings('ignore')

# CANCER CONFIGURATION: the only lines to edit for a new cancer
CANCER     = "LUAD"
base_dir   = r"causal_inf_new"

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, CANCER.lower())

# this cancer's expression / mutation files (written by 01 and 02)
data_dir   = os.path.join(cancer_dir, f"{CANCER.lower()}_data")
prefix     = CANCER.lower()
output_dir = os.path.join(cancer_dir, f"{prefix}_output")

# GO files (go-basic.obo, goa_human.gaf) are shared across cancers
GO_DIR = r"GBM_new"

# Independent cohort used for co-expression validation: cancer -> (name, folder inside base_dir)
VALIDATION_COHORTS = {"BRCA": ("METABRIC", "brca_metabric"), "KIRC": ("EMTAB1980", "kirc_emtab1980"),
                      "LUAD": ("GSE72094", "luad_gse72094")}
VAL_NAME, val_folder = VALIDATION_COHORTS[CANCER]

# file prefix: "VAL_NAME" -> same file names as before
val     = VAL_NAME.lower()                       
val_dir = os.path.join(data_dir, val_folder)

# CPU workers for the null-distribution step
N_JOBS = 16

# Other necessary paths
miner_dir  = os.path.dirname(miner.__file__)
idmap_path = os.path.join(miner_dir, "data", "identifier_mappings.txt")
godag = GODag(os.path.join(GO_DIR, "go-basic.obo"))
print(f"Cancer: {CANCER} | validation cohort: {VAL_NAME} | GO DAG: {len(godag)} terms")

GBM_new\go-basic.obo: fmt(1.2) rel(2026-03-25) 41,853 Terms
Cancer: LUAD | validation cohort: GSE72094 | GO DAG: 41853 terms


In [2]:
# Load regulon eigengenes: PC1 activity score per regulon per patient.
# Produced already in 02_regulons_clusters_states.ipynb.
eigengenes = pd.read_csv(os.path.join(output_dir, "eigengenes.csv"), index_col=0)
eigengenes.columns = eigengenes.columns.astype(str)

print(f"eigengenes shape: {eigengenes.shape}")
print(f"regulon id sample: {eigengenes.index[:5].tolist()}")
print(f"patient id sample: {eigengenes.columns[:3].tolist()}")

eigengenes shape: (7447, 516)
regulon id sample: [0, 1, 2, 3, 4]
patient id sample: ['TCGA-64-5779', 'TCGA-86-8073', 'TCGA-55-1594']


In [3]:
# Load clinical data (survival status and time) saved by 03
clinical = pd.read_csv(os.path.join(output_dir, "clinical_data.csv"), index_col=0)
print(f"Loaded clinical_data.csv from 03: {clinical.shape}")

Loaded clinical_data.csv from 03: (513, 2)


In [4]:
# Align to patients present in both eigengenes and clinical data.
common_patients  = list(set(eigengenes.columns) & set(clinical.index))
eigen_aligned    = eigengenes.loc[:, common_patients].T   # patients x regulons
clinical_aligned = clinical.loc[common_patients, ["OS_MONTHS", "OS_STATUS"]]

print(f"eigengenes patients: {eigengenes.shape[1]}")
print(f"clinical patients:   {clinical.shape[0]}")
print(f"common patients:     {len(common_patients)}")

eigengenes patients: 516
clinical patients:   513
common patients:     507


In [5]:
# Sanity check on a single regulon before batch fitting
# CoxPHFitter regresses time-to-event (OS_MONTHS) against the regulon's eigengene value,
# censoring on OS_STATUS (1 = death observed, 0 = censored/alive at last follow-up).
# HR > 1 means higher eigengene activity associates with worse (shorter) survival.
test_regulon = eigen_aligned.columns[0]

df_test = clinical_aligned.copy()
df_test["eigengene"] = eigen_aligned[test_regulon].values
df_test = df_test.dropna()

cph = CoxPHFitter()
cph.fit(df_test, duration_col="OS_MONTHS", event_col="OS_STATUS")
print(f"Regulon {test_regulon}: HR={cph.hazard_ratios_['eigengene']:.4f}, p={cph.summary.loc['eigengene','p']:.4f}")

Regulon 0: HR=0.8301, p=0.0163


In [6]:
# Batch Cox test across every regulon, 500 at a time 
cox_results = []
regulon_list = eigen_aligned.columns.tolist()
batch_size = 500

for batch_start in range(0, len(regulon_list), batch_size):
    batch = regulon_list[batch_start:batch_start + batch_size]
    for regulon in batch:
        df_cox = clinical_aligned.copy()
        df_cox["eigengene"] = eigen_aligned[regulon].values
        df_cox = df_cox.dropna()
        try:
            cph = CoxPHFitter()
            cph.fit(df_cox, duration_col="OS_MONTHS", event_col="OS_STATUS")
            hr = cph.hazard_ratios_["eigengene"]
            p  = cph.summary.loc["eigengene", "p"]
            cox_results.append({"regulon": regulon, "HR": hr, "p_value": p})
        except Exception:
            cox_results.append({"regulon": regulon, "HR": np.nan, "p_value": np.nan})
    print(f"completed {min(batch_start + batch_size, len(regulon_list))}/{len(regulon_list)} regulons")

cox_df = pd.DataFrame(cox_results).set_index("regulon")

completed 500/7447 regulons
completed 1000/7447 regulons
completed 1500/7447 regulons
completed 2000/7447 regulons
completed 2500/7447 regulons
completed 3000/7447 regulons
completed 3500/7447 regulons
completed 4000/7447 regulons
completed 4500/7447 regulons
completed 5000/7447 regulons
completed 5500/7447 regulons
completed 6000/7447 regulons
completed 6500/7447 regulons
completed 7000/7447 regulons
completed 7447/7447 regulons


In [7]:
# Regulons here are disease-relevant but only based on survival. Have not been checked against other criteria
# Filter to p-val threshold

cox_df["disease_relevant"] = cox_df["p_value"] <= 0.05
cox_df.to_csv(os.path.join(output_dir, "regulon_cox_results.csv"))

# Save outputs
disease_relevant = cox_df[cox_df["disease_relevant"]].index.tolist()
pd.Series(disease_relevant).to_csv(
    os.path.join(output_dir, "disease_relevant_regulons_survival_only.csv"),
    index=False, header=False
)

print(f"total regulons tested: {len(cox_df)}")
print(f"survival-significant (p<=0.05): {len(disease_relevant)}")
print(f"high risk (HR>1, p<=0.05): {((cox_df['HR']>1) & cox_df['disease_relevant']).sum()}")
print(f"low risk  (HR<1, p<=0.05): {((cox_df['HR']<1) & cox_df['disease_relevant']).sum()}")

total regulons tested: 7447
survival-significant (p<=0.05): 2919
high risk (HR>1, p<=0.05): 2085
low risk  (HR<1, p<=0.05): 834


In [8]:
# Download the GO annotation file (human gene -> GO term) if it isn't already in GO_DIR
gbm_dir = GO_DIR

gaf_path = os.path.join(gbm_dir, "goa_human.gaf.gz")
if not os.path.exists(gaf_path):
    resp = requests.get(
        "https://current.geneontology.org/annotations/goa_human.gaf.gz",
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=60,
    )
    resp.raise_for_status()
    with open(gaf_path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content)/1e6:.1f} MB")
else:
    print("Reusing existing goa_human.gaf.gz")

Reusing existing goa_human.gaf.gz


In [9]:
# Unzip the annotation file (goatools reads the plain .gaf)
gaf_plain_path = os.path.join(gbm_dir, "goa_human.gaf")

with gzip.open(gaf_path, "rt", encoding="utf-8", errors="replace") as fin, \
     open(gaf_plain_path, "w", encoding="cp1252", errors="replace") as fout:
    for line in fin:
        fout.write(line)

print(f"Wrote {gaf_plain_path}")

Wrote GBM_new\goa_human.gaf


In [10]:
# Read the annotations and keep the Biological Process (BP) namespace
gafreader = GafReader(gaf_plain_path)
ns2assoc = gafreader.get_ns2assc()
bp = ns2assoc["BP"]
sample_key = next(iter(bp))
print(f"BP-annotated genes: {len(bp)}")
print(f"sample key: {sample_key!r}")

HMS:0:00:18.484734 906,445 annotations READ: GBM_new\goa_human.gaf 
BP-annotated genes: 31530
sample key: 'A0A075B6T8'


In [11]:
# Inspect one raw annotation row to find the field holding the gene symbol (DB_Symbol)
raw_assocs = gafreader.associations
print(f"total raw association rows: {len(raw_assocs)}")

first = raw_assocs[0]
print(type(first))
print(first._fields)   # namedtuple field names — look for something like DB_Symbol / DB_Object_Symbol
print(first)

total raw association rows: 906445
<class 'goatools.anno.init.reader_gaf.ntgafobj'>
('DB', 'DB_ID', 'DB_Symbol', 'Qualifier', 'GO_ID', 'DB_Reference', 'Evidence_Code', 'With_From', 'NS', 'DB_Name', 'DB_Synonym', 'DB_Type', 'Taxon', 'Date', 'Assigned_By', 'Extension', 'Gene_Product_Form_ID')
ntgafobj(DB='UniProtKB', DB_ID='A0A024RBG1', DB_Symbol='NUDT4B', Qualifier={'enables'}, GO_ID='GO:0005515', DB_Reference={'PMID:33961781'}, Evidence_Code='IPI', With_From={'UniProtKB:Q8NFP7'}, NS='MF', DB_Name={'Diphosphoinositol polyphosphate phosphohydrolase NUDT4B'}, DB_Synonym={'NUDT4B'}, DB_Type='protein', Taxon=[9606], Date=datetime.date(2026, 4, 25), Assigned_By='IntAct', Extension=None, Gene_Product_Form_ID={'UniProtKB:A0A024RBG1'})


In [12]:
# Gene symbol -> set of BP GO terms, skipping negative ('NOT') annotations
symbol2bp_go = defaultdict(set)

for row in raw_assocs:
    if row.NS != "BP":
        continue
    if any("NOT" in q for q in row.Qualifier):
        continue
    symbol2bp_go[row.DB_Symbol].add(row.GO_ID)

symbol2bp_go = dict(symbol2bp_go)
print(f"genes with >=1 BP annotation (by symbol): {len(symbol2bp_go)}")

genes with >=1 BP annotation (by symbol): 31557


In [13]:
# Regulon genes: Ensembl -> symbol, and how many have at least one BP annotation
# (exploratory; the next cell repeats the steps that are actually needed)
with open(os.path.join(output_dir, "regulons.json")) as f:
    regulons = json.load(f)

regulon_genes = set()
for genes in regulons.values():
    regulon_genes.update(genes)

print(f"total unique genes across all {CANCER} regulons: {len(regulon_genes)}")

idmap = pd.read_csv(idmap_path, sep="\t", header=0)  # same idmap_path your notebook 02 already defines
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

idmap = pd.read_csv(idmap_path, sep="\t", header=0)
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

regulons_symbol = {
    tf: [ensembl_to_symbol.get(g, g) for g in genes]
    for tf, genes in regulons.items()
}

regulon_genes_symbol = set()
for genes in regulons_symbol.values():
    regulon_genes_symbol.update(genes)

print(f"unique regulon genes (symbol): {len(regulon_genes_symbol)}")
covered = regulon_genes_symbol & symbol2bp_go.keys()
print(f"with >=1 BP annotation: {len(covered)} / {len(regulon_genes_symbol)} ({100*len(covered)/len(regulon_genes_symbol):.1f}%)")
print(f"unique regulon genes (symbol): {len(regulon_genes_symbol)}")
covered = regulon_genes_symbol & symbol2bp_go.keys()
print(f"with >=1 BP annotation: {len(covered)} / {len(regulon_genes_symbol)} ({100*len(covered)/len(regulon_genes_symbol):.1f}%)")

covered = regulon_genes & symbol2bp_go.keys()
missing = regulon_genes - symbol2bp_go.keys()

pct = 100 * len(covered) / len(regulon_genes)
print(f"regulon genes with >=1 BP annotation: {len(covered)} / {len(regulon_genes)} ({pct:.1f}%)")
print(f"sample of missing genes: {sorted(missing)[:20]}")

total unique genes across all LUAD regulons: 11747
unique regulon genes (symbol): 11747
with >=1 BP annotation: 10140 / 11747 (86.3%)
unique regulon genes (symbol): 11747
with >=1 BP annotation: 10140 / 11747 (86.3%)
regulon genes with >=1 BP annotation: 0 / 11747 (0.0%)
sample of missing genes: ['ENSG00000000003', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000938', 'ENSG00000001036', 'ENSG00000001084', 'ENSG00000001167', 'ENSG00000001460', 'ENSG00000001461', 'ENSG00000001617', 'ENSG00000001626', 'ENSG00000001629', 'ENSG00000001630', 'ENSG00000002016', 'ENSG00000002330', 'ENSG00000002549', 'ENSG00000002587', 'ENSG00000002745', 'ENSG00000002746', 'ENSG00000002834']


In [14]:
# Everything the GO enrichment needs, in one cell: regulons as symbols, symbol -> BP terms, background genes, GO DAG
idmap = pd.read_csv(idmap_path, sep="\t", header=0)
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

with open(os.path.join(output_dir, "regulons.json")) as f:
    regulons = json.load(f)

regulons_symbol = {tf: [ensembl_to_symbol.get(g, g) for g in genes] for tf, genes in regulons.items()}
regulon_genes_symbol = set(g for genes in regulons_symbol.values() for g in genes)
print(f"unique regulon genes (symbol): {len(regulon_genes_symbol)}")

gaf_plain_path = os.path.join(GO_DIR, "goa_human.gaf")

gafreader = GafReader(gaf_plain_path)
raw_assocs = gafreader.associations

symbol2bp_go = defaultdict(set)
for row in raw_assocs:
    if row.NS != "BP":
        continue
    if any("NOT" in q for q in row.Qualifier):
        continue
    symbol2bp_go[row.DB_Symbol].add(row.GO_ID)
symbol2bp_go = dict(symbol2bp_go)

print(f"genes with >=1 BP annotation (by symbol): {len(symbol2bp_go)}")

# Background = every gene expressed in this cancer (not just regulon genes)
background_genes = set(pd.read_csv(os.path.join(data_dir, f"{prefix}_zscored.csv"), index_col=0, usecols=[0]).index)
print(f"full {CANCER} expressed gene universe: {len(background_genes)}")

godag = GODag(os.path.join(GO_DIR, "go-basic.obo"))
print(f"GO DAG loaded: {len(godag)} terms")

unique regulon genes (symbol): 11747
HMS:0:00:19.881648 906,445 annotations READ: GBM_new\goa_human.gaf 
genes with >=1 BP annotation (by symbol): 31557
full LUAD expressed gene universe: 19734
GBM_new\go-basic.obo: fmt(1.2) rel(2026-03-25) 41,853 Terms
GO DAG loaded: 41853 terms


In [15]:
# ORA (overrepresentation analysis) of GO enrichment terms. Answers the question "out of all GO terms this regulon's genes are
# annotated with, which occur signficantly more than by chance?"

# Calculates 4 values: how how many regulon genes have GO term X, how many regulon genes are there in total, how many background genes have term X 
# and how many background genes are there in total

# These 4 terms give raw p-value for each GO term. After correction for FDR, only significant terms remain

# Builds the background side of the ORA analysis so that it doesn't have to be calculated repeatedly:
goe_obj = GOEnrichmentStudy(

    # checks if this gene set is enriched with GO terms
    background_genes,

    # dict of the format {str: list} where key is HUGO symbol and value is [GO term 1, GO term 2, ...]
    symbol2bp_go,

    # DAG (directed acyclic graph) of GO terms w/ child-parent rels. E.g.: positive regulation of process X is a child term of process X
    godag,

    # Propagates up the DAG (basically if a gene is annoted with a child term, it is marked as annoted with parent terms all the way up the DAG)
    propagate_counts=True,

    # Significance threshold; p_val must be <= alpha
    alpha=0.05,

    # Benjamini-Hochberg correction for FDR
    methods=['fdr_bh']
)


Load  Ontology Enrichment Analysis ...
Propagating term counts up: is_a
 81% 16,072 of 19,734 population items found in association


In [16]:
# Helper fn: tracks GO terms in a particular geneset
def study_terms(study_genes, assoc):

    '''

    Collects every GO term that at least one gene of a gene set is annotated with.

    Args: study_genes (list). Gene symbols of one regulon
          assoc (dict). {gene symbol: set of GO term IDs}

    Returns: terms (set) GO term IDs annotated to any gene in study_genes

    '''

    terms = set()

    # Union of GO terms in each set
    for g in study_genes:
        terms |= assoc.get(g, set())
    return terms

# Helper fn: FDR correction with m (no.of genes in background) = total no. of terms across all genes not just regulon genes
def bh_adjust_with_true_m(pvals, m_true):

    '''

    Benjamini-Hochberg FDR correction where the number of tests m is given explicitly (all GO terms in the background)
    instead of the number of p-values passed in (only the terms the regulon's genes carry).

    Args: pvals (list), length n_terms_tested. Raw p-values
          m_true (int). Number of GO terms in the whole background population

    Returns: out (np.array), shape (n_terms_tested,). q-values in the same order as pvals

    '''

    # Sort p-values in ascending order
    order = np.argsort(pvals)
    ranked = np.array(pvals)[order]

    # BH: q = p * m / rank
    q = ranked * m_true / (np.arange(len(ranked)) + 1)

    # Enforce monotonicity: a larger p-value can't get a smaller q-value
    # Then cap at 1
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)

    # Put q-values back in the original order
    out = np.empty_like(q)
    out[order] = q
    return out

# Setup
checkpoint_every = 500
checkpoint_path = os.path.join(output_dir, "go_bio_symbol_checkpoint.pkl")
final_path      = os.path.join(output_dir, "go_bio_symbol.pkl")

# Takes ~45 min, so reuse the saved result if it exists (delete go_bio_symbol.pkl to recompute)
if os.path.exists(final_path):
    with open(final_path, "rb") as f:
        go_bio_symbol = pickle.load(f)
    print(f"Loaded cached go_bio_symbol.pkl: {len(go_bio_symbol)} regulons with significant BP terms")

else:
    go_bio_symbol = {}
    m_true = len(goe_obj.go2popitems)
    print(f"m_true (population term count): {m_true}")
    print(f"regulons to process: {len(regulons_symbol)}")

    start = time.time()

    # Main loop:
    for i, (regulon_id, genes) in enumerate(regulons_symbol.items()):
        # Regulon genes present in the background; regulons with < 3 are skipped
        study_genes = [g for g in genes if g in background_genes]
        if len(study_genes) < 3:
            continue

        # Only test GO terms that at least one regulon gene is annotated with
        sel_terms = study_terms(study_genes, symbol2bp_go)
        if not sel_terms:
            continue

        results = goe_obj.run_study(study_genes, prt=None, selected_goids=sel_terms)
        if not results:
            continue

        # FDR-correct against the full background term count and keep terms with q <= 0.05
        raw_p = [r.p_uncorrected for r in results]
        qvals = bh_adjust_with_true_m(raw_p, m_true)
        sig_terms = [r.GO for r, q in zip(results, qvals) if q <= 0.05]
        if sig_terms:
            go_bio_symbol[regulon_id] = sig_terms

        if (i + 1) % checkpoint_every == 0:
            elapsed = time.time() - start
            print(f"{i+1}/{len(regulons_symbol)} done, {elapsed/60:.1f} min elapsed, {len(go_bio_symbol)} regulons with sig terms so far")
            with open(checkpoint_path, "wb") as f:
                pickle.dump(go_bio_symbol, f)

    # Save:
    print(f"DONE: {len(go_bio_symbol)} / {len(regulons_symbol)} regulons with significant BP terms, total time {(time.time()-start)/60:.1f} min")
    with open(final_path, "wb") as f:
        pickle.dump(go_bio_symbol, f)

m_true (population term count): 13467
regulons to process: 7447
500/7447 done, 3.4 min elapsed, 93 regulons with sig terms so far
1000/7447 done, 6.8 min elapsed, 194 regulons with sig terms so far
1500/7447 done, 9.9 min elapsed, 284 regulons with sig terms so far
2000/7447 done, 12.6 min elapsed, 370 regulons with sig terms so far
2500/7447 done, 15.0 min elapsed, 439 regulons with sig terms so far
3000/7447 done, 17.7 min elapsed, 525 regulons with sig terms so far
3500/7447 done, 20.0 min elapsed, 581 regulons with sig terms so far
4000/7447 done, 22.4 min elapsed, 635 regulons with sig terms so far
4500/7447 done, 24.7 min elapsed, 688 regulons with sig terms so far
5000/7447 done, 26.7 min elapsed, 716 regulons with sig terms so far
5500/7447 done, 28.8 min elapsed, 744 regulons with sig terms so far
6000/7447 done, 30.6 min elapsed, 771 regulons with sig terms so far
6500/7447 done, 32.4 min elapsed, 796 regulons with sig terms so far
7000/7447 done, 33.9 min elapsed, 817 regulo

In [17]:
# 10 hallmarks of cancer (Hanahan-Weinberg)
# Rough search; pulls out all terms in the GO DAG that contain certain keywords related to each hallmark

hallmarks_updated = {
    'SelfSufficiencyInGrowthSignals':      {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['prolifer', 'growth factor', 'mitogen', 'cell growth'])},
    'InsensitivityToAntigrowthSignals':    {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['cell cycle', 'growth inhibit', 'antiproliferat', 'cyclin', 'tumor suppress'])},
    'EvadingApoptosis':                    {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['apoptot', 'apoptosis', 'programmed cell death', 'survival'])},
    'LimitlessReplicativePotential':       {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['telomer', 'replicat', 'immortal', 'cell division'])},
    'SustainedAngiogenesis':               {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['angiogenes', 'vasculogenes', 'vessel', 'vascular'])},
    'TissueInvasionAndMetastasis':         {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['invasion', 'metastas', 'migration', 'motility', 'extracellular matrix'])},
    'GenomeInstabilityAndMutation':        {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['dna repair', 'genome instab', 'mutat', 'dna damage', 'chromosome instab'])},
    'TumorPromotingInflammation':          {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['inflammat', 'immune respon', 'cytokine', 'nf-kb', 'interleukin'])},
    'ReprogrammingEnergyMetabolism':       {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['glycolys', 'oxidative phosphorylat', 'metabol', 'atp', 'glucose'])},
    'EvadingImmuneDetection':              {t for t in godag if godag[t].namespace == 'biological_process' and any(k in godag[t].name.lower() for k in ['immune evasion', 'immune escape', 'immunosuppress', 't cell', 'mhc'])},
}

# Show the number of GO terms associated w/ each of the 10 hallmarks
for h, terms in hallmarks_updated.items():
    print(f"{h}: {len(terms)} GO terms")

# Computes term rarity based on how many regulons are enriched for each term
term_counts = TermCounts(godag, go_bio_symbol)  

# Every unique GO term that the regulons are enriched for
regulon_go_terms  = list({t for terms in go_bio_symbol.values() for t in terms if t in godag})

# Every unique GO term associated with the cancer hallmarks
hallmark_go_terms = list({t for terms in hallmarks_updated.values() for t in terms if t in godag})

print(f"Unique regulon GO terms:  {len(regulon_go_terms)}")
print(f"Unique hallmark GO terms: {len(hallmark_go_terms)}")

SelfSufficiencyInGrowthSignals: 619 GO terms
InsensitivityToAntigrowthSignals: 205 GO terms
EvadingApoptosis: 380 GO terms
LimitlessReplicativePotential: 330 GO terms
SustainedAngiogenesis: 197 GO terms
TissueInvasionAndMetastasis: 424 GO terms
GenomeInstabilityAndMutation: 75 GO terms
TumorPromotingInflammation: 852 GO terms
ReprogrammingEnergyMetabolism: 1207 GO terms
EvadingImmuneDetection: 707 GO terms
Unique regulon GO terms:  2295
Unique hallmark GO terms: 4714


In [18]:
# Now doing Lin semantic similarity: for every term in regulon_go_terms, it asks how similar it is to the terms in hallmark_go_terms
# Returns a score b/w 0 (fully unrelated) and 1 (identical)

# Prints output every x interactions
checkpoint_every = 250

checkpoint_path = os.path.join(output_dir, "lin_cache_checkpoint.pkl")
final_path      = os.path.join(output_dir, "lin_cache.pkl")

# Takes ~15 min for BRCA, so reuse the saved result if it exists (delete lin_cache.pkl to recompute)
if os.path.exists(final_path):
    with open(final_path, "rb") as f:
        lin_cache = pickle.load(f)
    print(f"Loaded cached lin_cache.pkl: {len(lin_cache)} regulon terms")

else:
    # Lookup table
    lin_cache = {}
    start = time.time()

    # Goes through each of the regulon terms
    for i, t1 in enumerate(regulon_go_terms):

        sims = {}

        # For each regulon term, it goes through each of the hallmark terms
        for t2 in hallmark_go_terms:

            try:
                # Gets similarity score between the current t1 (regulon term) and current t2 (hallmark term)
                # term_counts contains term rarities based on how many regulons are enriched
                s = lin_sim(t1, t2, godag, term_counts)

                # Safety net; if similarity returns None, defaults to 0
                sims[t2] = s if s else 0

            # Safety net but if similarity fails
            except:
                sims[t2] = 0

        # sims is a dict of the form {hallmark term: similarity score} for the current regulon term (t1)
        lin_cache[t1] = sims

        # Periodically displays progress and saves for later
        if (i + 1) % checkpoint_every == 0:
            elapsed = time.time() - start
            print(f"{i+1}/{len(regulon_go_terms)} done, {elapsed/60:.1f} min elapsed")
            with open(checkpoint_path, "wb") as f:
                pickle.dump(lin_cache, f)

    # Finally, lin_cache is a nested dict of the format {regulon term : {hallmark term 1: sim score 1, hallmark term 2: sim score 2, ...}}
    print(f"DONE: {len(lin_cache)} regulon terms cached, total time {(time.time()-start)/60:.1f} min")
    with open(final_path, "wb") as f:
        pickle.dump(lin_cache, f)

250/2295 done, 0.8 min elapsed
500/2295 done, 1.4 min elapsed
750/2295 done, 2.0 min elapsed
1000/2295 done, 2.7 min elapsed
1250/2295 done, 3.3 min elapsed
1500/2295 done, 3.9 min elapsed
1750/2295 done, 4.5 min elapsed
2000/2295 done, 5.0 min elapsed
2250/2295 done, 5.7 min elapsed
DONE: 2295 regulon terms cached, total time 5.8 min


In [19]:
# Takes one regulon id, its list of sig GO terms and a similarity threshold (here it is 80%)
# regulon_go_terms[regulon_id] = enriched_go
def regulon_hallmark_sim_fast(reg_id, enriched_go, threshold=0.8):

    '''

    Finds which cancer hallmarks a regulon is enriched for. A hallmark is passed if any of the regulon's GO terms has
    Lin similarity >= threshold with any GO term of that hallmark. Uses lin_cache, hallmarks_updated and godag from above.

    Args: reg_id (str). Regulon ID
          enriched_go (list). Significant GO term IDs of the regulon
          threshold (float). Minimum Lin similarity to count as a match (default 0.8)

    Returns: reg_id (str), passed (list) names of the hallmarks the regulon qualifies for

    '''

    # collects hallmark CATEGORIES that the regulon qualifies for
    passed = []

    # Drops any GO terms not present in lin_cache, which contains all GO terms for which regulon is enriched
    enriched_go = [t for t in enriched_go if t in lin_cache]

    # Loops through each of the 10 Hanahan-Weinberg hallmarks
    # hallmarks_updated is dict of the form {hallmark: [go_terms]}
    for hallmark, go_terms in hallmarks_updated.items():

        # Safety check: only takes hallmark terms present in the overall GO DAG
        hallmark_terms = [t for t in go_terms if t in godag]

        
        max_sim = 0

        # Highest similarity between any regulon term and any term of this hallmark (stops early once it hits 1.0)
        for t1 in enriched_go:
            for t2 in hallmark_terms:
                s = lin_cache[t1].get(t2, 0)
                if s > max_sim:
                    max_sim = s
                if max_sim == 1.0:
                    break
            if max_sim == 1.0:
                break
        if max_sim >= threshold:
            passed.append(hallmark)
    return reg_id, passed

# Apply to every regulon with significant GO terms: {regulon ID: [hallmarks passed]}
hallmark_enriched = {}
for reg_id, go_terms in go_bio_symbol.items():
    if len(go_terms) == 0:
        continue
    _, passed = regulon_hallmark_sim_fast(reg_id, list(go_terms))
    if passed:
        hallmark_enriched[reg_id] = passed

print("Regulons with hallmark enrichment:", len(hallmark_enriched))

all_hallmarks = [h for hallmarks in hallmark_enriched.values() for h in hallmarks]
print(Counter(all_hallmarks).most_common())

Regulons with hallmark enrichment: 663
[('ReprogrammingEnergyMetabolism', 353), ('TissueInvasionAndMetastasis', 340), ('TumorPromotingInflammation', 336), ('SelfSufficiencyInGrowthSignals', 245), ('EvadingImmuneDetection', 212), ('SustainedAngiogenesis', 204), ('InsensitivityToAntigrowthSignals', 167), ('GenomeInstabilityAndMutation', 145), ('LimitlessReplicativePotential', 133), ('EvadingApoptosis', 101)]


In [20]:
# Combine the two criteria: survival-significant OR hallmark-enriched
survival_series = pd.read_csv(os.path.join(output_dir, "disease_relevant_regulons_survival_only.csv"), header=None)[0]
survival_set = set(survival_series.astype(str))
hallmark_set = set(str(k) for k in hallmark_enriched.keys())

overlap = survival_set & hallmark_set
survival_or_hallmark = survival_set | hallmark_set

print(f"Survival significant: {len(survival_set)}")
print(f"Hallmark enriched: {len(hallmark_set)}")
print(f"Overlap (both criteria): {len(overlap)}")
print(f"Survival OR hallmark (final disease-relevant set): {len(survival_or_hallmark)}")

pd.Series(sorted(survival_or_hallmark, key=int)).to_csv(
    os.path.join(output_dir, "final_disease_relevant_regulons.csv"), index=False, header=False)

print(f"Saved {len(survival_or_hallmark)} disease-relevant regulons")

Survival significant: 2919
Hallmark enriched: 663
Overlap (both criteria): 316
Survival OR hallmark (final disease-relevant set): 3266
Saved 3266 disease-relevant regulons


In [21]:
# Keep the intermediate (survival OR hallmark) set in its own file.
# final_disease_relevant_regulons.csv gets overwritten further down, once the VAL_NAME co-expression validation is done
# (GBM equivalent: final = (survival OR hallmark) AND Gravendeel co-expression validated)
pd.Series(sorted(survival_or_hallmark, key=int)).to_csv(
    os.path.join(output_dir, "disease_relevant_regulons_survival_or_hallmark.csv"),
    index=False, header=False
)
print(f"Saved {len(survival_or_hallmark)} survival-OR-hallmark regulons")

Saved 3266 survival-OR-hallmark regulons


## Co-expression validation in an independent cohort (VAL_NAME)

GBM used Gravendeel for this step; for each cancer the independent cohort is **VAL_NAME** (in this case, GSE72094).

A regulon counts as validated if its genes are still co-expressed in VAL_NAME:
1. PC1 of the regulon's genes explains more variance than 95% of random gene sets of the same size (permutation p ≤ 0.05), **and**
2. PC1 variance explained ≥ 0.3 (hard cutoff from the paper)

The final disease-relevant set is defined at the end of this notebook, under the paper's rule.

In [22]:
# Load the validation cohort: expression with one row per gene symbol, plus clinical data if available
# Formats handled: cBioPortal studies (e.g. METABRIC), and GSE72094 (GEO SOFT file)
# For any other cohort, add a branch that builds mb_raw (gene symbols x samples, log2 scale) and, if available, mb_clin

metabric_dir = val_dir
mb_clin = None


if VAL_NAME == "GSE72094":

    # GEO "SOFT family" file: one platform block (probe ID -> gene symbol) followed by one block per sample,
    # each with clinical fields (!Sample_characteristics_ch1 = name: value) and a table of probe ID / log2 expression
    # Parsing the 250 MB file takes a few minutes, so the two tidy tables are cached next to it (delete them to rebuild)
    soft_file = os.path.join(metabric_dir, "GSE72094_family.soft.gz")
    expr_csv  = os.path.join(metabric_dir, "gse72094_expression_by_symbol.csv.gz")
    clin_csv  = os.path.join(metabric_dir, "gse72094_clinical_raw.csv")

    if not (os.path.exists(expr_csv) and os.path.exists(clin_csv)):
        assert os.path.exists(soft_file), f"missing {soft_file}"
        probe_ids, probe_to_symbol, probe_pos, samples, clin_rows = [], {}, None, {}, {}
        block, in_table, sample_id = None, False, None

        with gzip.open(soft_file, "rt", errors="replace") as f:
            for line in f:
                line = line.rstrip("\n")

                # A line starting with ^ opens a new block (platform, or one sample)
                if line.startswith("^"):
                    block, in_table = line.split(" = ")[0], False
                    if block == "^SAMPLE":
                        # The platform block comes first, so the full probe list is known by now
                        # One float32 array per sample (position = probe) keeps memory low: ~60k probes x 442 samples
                        if probe_pos is None:
                            probe_pos = {p: i for i, p in enumerate(probe_ids)}
                        sample_id = line.split(" = ")[1]
                        samples[sample_id] = np.full(len(probe_ids), np.nan, dtype=np.float32)
                        clin_rows[sample_id] = {}
                    continue

                # Table boundaries inside a block
                if line.startswith("!platform_table_begin") or line.startswith("!sample_table_begin"):
                    in_table, header_seen = True, False
                    continue
                if line.startswith("!platform_table_end") or line.startswith("!sample_table_end"):
                    in_table = False
                    continue

                if in_table:
                    # First row of every table is its header
                    if not header_seen:
                        header_seen = True
                        continue
                    parts = line.split("\t")

                    # Platform table columns: ID, GB_LIST, EntrezGeneID, GeneSymbol, SPOT_ID
                    if block == "^PLATFORM":
                        probe_ids.append(parts[0])
                        if len(parts) >= 4 and parts[3].strip():
                            probe_to_symbol[parts[0]] = parts[3].strip()

                    # Sample table columns: ID_REF, VALUE
                    elif block == "^SAMPLE" and len(parts) >= 2 and parts[0] in probe_pos:
                        try:
                            samples[sample_id][probe_pos[parts[0]]] = float(parts[1])
                        except ValueError:
                            pass

                # Clinical fields of the current sample
                elif block == "^SAMPLE" and line.startswith("!Sample_characteristics_ch1 = ") and ": " in line:
                    key, value = line[len("!Sample_characteristics_ch1 = "):].split(": ", 1)
                    clin_rows[sample_id][key.strip()] = value.strip()

        # Probes x samples, keeping probes that map to exactly one gene symbol, then one row per symbol (mean of its probes)
        probes = pd.DataFrame(samples, index=probe_ids)
        del samples
        symbols = pd.Series(probe_to_symbol).reindex(probes.index)
        single = symbols.notna() & ~symbols.fillna("").str.contains(r"[ /,;|]")
        print(f"Probes mapped to a single gene symbol: {int(single.sum())} / {len(probes)}")
        by_symbol = probes[single].groupby(symbols[single]).mean()
        by_symbol.index.name = "Hugo_Symbol"
        by_symbol.to_csv(expr_csv)
        pd.DataFrame(clin_rows).T.rename_axis("PATIENT_ID").to_csv(clin_csv)

    mb_raw = pd.read_csv(expr_csv, index_col=0)

    # Overall survival: vital_status (Alive / Dead) and survival_time_in_days; patients with either missing are dropped
    clin_geo = pd.read_csv(clin_csv, index_col=0)
    mb_clin = pd.DataFrame({
        "OS_MONTHS": pd.to_numeric(clin_geo["survival_time_in_days"], errors="coerce") / 30.4375,
        "OS_STATUS": clin_geo["vital_status"].astype(str).str.strip().str.lower().map({"alive": 0, "dead": 1}),
    }).dropna()
    mb_clin = mb_clin[mb_clin["OS_MONTHS"] >= 0]
    mb_clin["OS_STATUS"] = mb_clin["OS_STATUS"].astype(int)

else:

    # cBioPortal format: extract all relevant mRNA expression files
    expr_files = [f for f in glob.glob(os.path.join(metabric_dir, "**", "data_mrna*.txt"), recursive=True)
                  if "zscore" not in os.path.basename(f).lower()]

    print("Expression file:", expr_files)
    assert expr_files, f"no data_mrna*.txt found in {metabric_dir}"

    mb_raw = pd.read_csv(expr_files[0], sep="\t")

    # Drop extra gene symbol column (Entrez ID)
    mb_raw = mb_raw.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in mb_raw.columns])

    # Collapse duplicate symbols
    mb_raw = mb_raw.dropna(subset=["Hugo_Symbol"]).groupby("Hugo_Symbol").mean(numeric_only=True)

    # Load clinical data; header lines starting with '#' are dropped
    clin_file = glob.glob(os.path.join(metabric_dir, "**", "data_clinical_patient.txt"), recursive=True)[0]

    # Keep all columns for now (needed for disease-specific survival below), then extract the OS columns
    mb_clin_all = pd.read_csv(clin_file, sep="\t", comment="#")
    mb_clin = mb_clin_all[["PATIENT_ID", "OS_MONTHS", "OS_STATUS"]].copy()

    # Text to binary, dropping all NaN and negative survival time values
    mb_clin["OS_STATUS"] = pd.to_numeric(mb_clin["OS_STATUS"].astype(str).str.split(":").str[0], errors="coerce")
    mb_clin["OS_MONTHS"] = pd.to_numeric(mb_clin["OS_MONTHS"], errors="coerce")
    mb_clin = mb_clin.dropna()
    mb_clin = mb_clin[mb_clin["OS_MONTHS"] >= 0].set_index("PATIENT_ID")

    # Disease-specific survival when the cohort records cause of death (METABRIC: VITAL_STATUS)
    # Missing cause of death -> NaN (not 0), so a death with unknown cause isn't counted as censored
    if "VITAL_STATUS" in mb_clin_all.columns:
        vs = mb_clin_all.set_index("PATIENT_ID")["VITAL_STATUS"].reindex(mb_clin.index)
        mb_clin["DSS_STATUS"] = np.where(vs.isna(), np.nan, (vs == "Died of Disease").astype(float))
        print(f"Disease-specific deaths: {int(mb_clin['DSS_STATUS'].sum())} | unknown cause: {int(mb_clin['DSS_STATUS'].isna().sum())}")

print(f"{VAL_NAME} raw:", mb_raw.shape)

# Drop genes missing in >10% of samples, impute remaining NaNs with the mean gene exprn value
mb_raw = mb_raw.loc[mb_raw.isna().mean(axis=1) <= 0.10]
mb_raw = mb_raw.T.fillna(mb_raw.mean(axis=1)).T

# Z-score per gene
mb_std = mb_raw.std(axis=1)
mb_raw = mb_raw.loc[mb_std > 0]
mb_z   = ((mb_raw.T - mb_raw.mean(axis=1)) / mb_std[mb_std > 0]).T
print(f"{VAL_NAME} z-scored:", mb_z.shape, "| NaN:", int(mb_z.isna().sum().sum()))

# Saved for notebook 05 (pickle is much faster than a large csv)
mb_z.to_pickle(os.path.join(output_dir, f"{val}_zscored.pkl"))

if mb_clin is not None:
    # Diagnostic: should be close to 100% overlap
    print(f"{VAL_NAME} clinical: {mb_clin.shape} | OS events: {int(mb_clin['OS_STATUS'].sum())} | "
          f"overlap with expression: {len(set(mb_clin.index) & set(mb_z.columns))}")
    mb_clin.to_csv(os.path.join(output_dir, f"{val}_clinical.csv"))
else:
    print(f"{VAL_NAME}: no survival data found. Co-expression validation (this notebook) works without it; "
          f"notebook 05's external survival test needs {val}_clinical.csv")

GSE72094 raw: (21283, 442)
GSE72094 z-scored: (21283, 442) | NaN: 0
GSE72094 clinical: (398, 2) | OS events: 113 | overlap with expression: 398


In [23]:
# Map regulons (member genes have Ensembl IDs) to symbols and restrict to genes measured in VAL_NAME
idmap = pd.read_csv(idmap_path, sep="\t", header=0)
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))

with open(os.path.join(output_dir, "regulons.json")) as f:
    regulons = json.load(f)

mb_genes = set(mb_z.index)
regulon_modules_mb = {}
for reg_id, genes in regulons.items():
    syms = list(dict.fromkeys(ensembl_to_symbol.get(g, g) for g in genes))

    # Stores regulon ids that are present in validation cohort
    regulon_modules_mb[int(reg_id)] = [s for s in syms if s in mb_genes]

all_reg_genes = set(ensembl_to_symbol.get(g, g) for genes in regulons.values() for g in genes)
print(f"Regulon genes measured in {VAL_NAME}: {len(all_reg_genes & mb_genes)} / {len(all_reg_genes)}")

# PCA needs a few genes; regulons with <3 genes in VAL_NAME can't be tested and count as NOT validated
# Testable is dict of the format regulon_id: regulon_genes for all regulons in validation cohort with >= 3 genes
testable = {r: g for r, g in regulon_modules_mb.items() if len(g) >= 3}
print(f"Testable regulons (>=3 genes in {VAL_NAME}): {len(testable)} / {len(regulons)}")

# program_coherence_mb has columns:

# Modules (regulon_id)
# N (how many regulon genes in validation cohort),
# Variance (variance of all regulon genes within 1 patient, averaged over patients)
# Variance_expained (Share of variance explained by PC1)
program_coherence_mb = regulon_variance(mb_z, testable)

# regulon ids, in row order
coh_ids = program_coherence_mb.index.to_numpy()

# Drop index
program_coherence_mb = program_coherence_mb.reset_index(drop=True)      
print(program_coherence_mb.head())

Regulon genes measured in GSE72094: 10841 / 11747
Testable regulons (>=3 genes in GSE72094): 7442 / 7447
       N  Variance  Variance_expained  Modules
0  188.0  0.479685           0.549569      0.0
1   77.0  0.505643           0.505437      1.0
2   46.0  0.813745           0.218561      2.0
3   30.0  0.408359           0.605973      3.0
4   27.0  0.878979           0.173027      4.0


In [24]:
# Null distribution: 

# For each regulon size n, 250 groups of n genes from validation cohort are taken randomly and PC1 is computed
# If a given regulon's PC1 is higher than 95% of these gene sets (1- 0.95 = 0.05= p) and PC1 >= 0.3, it is coexpression-validated

# Differences from GBM (which made one serial call with n_iter=500):

# 250 draws per size
# 1 miner call per regulon size, run in parallel across CPU cores
# Reproducible: each size gets a fixed seed (SEED_BASE + size) before calling miner with seed=False,
# Results are identical on every rerun regardless of which core runs which size
# Cached to csv so it only runs once (delete the csv to recompute).

N_ITER    = 250
SEED_BASE = 12000

def _null_for_size(A, genes, n, n_iter, seed_base):

    '''

    Null distribution for one regulon size: draws n_iter random gene sets of n genes from the validation cohort and
    computes variance and PC1 variance explained for each. Runs in a separate worker process, hence the imports inside.

    Args: A (np.array), shape (n_genes, n_samples). Z-scored validation expression
          genes (list), length n_genes. Gene symbols, row labels of A
          n (int). Regulon size (number of genes per random set)
          n_iter (int). Number of random gene sets to draw
          seed_base (int). Random seed is seed_base + n, so each size is reproducible

    Returns: df, shape (n_iter, 3). Columns N, Variance, Variance_explained; one row per random gene set

    '''

    import numpy as np
    import pandas as pd
    from miner.miner import random_regulon_variance

    # A: entire expression matrix flattened into one array
    df = pd.DataFrame(A, index=genes)

    # Different random seed for each size, making it reproducible AND independent
    np.random.seed(seed_base + n)

    # Returns this df for each size : n 
    # Columns are variance, PC1 variance-explained
    # Shape: (1, 250) i.e (len([n], n_iter)
    return random_regulon_variance(df, [n], n_iter=n_iter, seed=False)

ns_list   = sorted(program_coherence_mb["N"].astype(int).unique())
rand_path = os.path.join(output_dir, f"{val}_random_regulon_variance.csv")
print(f"Distinct regulon sizes to simulate: {len(ns_list)} | draws per size: {N_ITER} | CPU cores: {os.cpu_count()}")

# If already run, load the cached null distribution
if os.path.exists(rand_path):
    random_results_mb = pd.read_csv(rand_path, index_col=0)
    print("Loaded cached null distribution:", random_results_mb.shape)

# Fresh run:
else:
    start = time.time()

    # Expression values as a plain float array (cheap to send to worker processes) and the gene names
    A     = np.ascontiguousarray(mb_z.values, dtype=np.float64)
    genes = list(mb_z.index)
    tasks = (delayed(_null_for_size)(A, genes, int(n), N_ITER, SEED_BASE)

             # largest (slowest) sizes first for load balancing
             for n in sorted(ns_list, reverse=True))   
    
    # Progress bar ticks once per regulon size; it starts slow (big sizes) and speeds up
    parts = list(tqdm(Parallel(n_jobs= N_JOBS, return_as="generator_unordered")(tasks),
                      total=len(ns_list), desc="regulon sizes"))

    # fixed order by size gives identical csv on rerun
    parts.sort(key=lambda d: int(d.index[0]))              
    random_results_mb = pd.concat(parts)
    random_results_mb.to_csv(rand_path)
    print(f"Null distribution done in {(time.time()-start)/60:.1f} min:", random_results_mb.shape)

# BOTH VARIANCE AND PC1 must be above 95% of null distbn. members. Below 2 lines calculate those thresholds
# Below 2 dfs are of the format row --> regulon size, column --> cutoff value of PC1/ variance
variance_explained_cutoffs = random_significance(random_results_mb, variable="Variance_explained", p=0.05)
variance_cutoffs = random_significance(random_results_mb, variable="Variance", p=0.05)

# Marks each regulon as significant or not (1/0) based on PC1, variance as compared to NULL DISTBN. not paper threshold
var_pass, var_exp_pass, var_passDF, var_exp_passDF = significant_regulons(
    program_coherence_mb, variance_cutoffs, variance_explained_cutoffs)

# Map row positions back to regulon ids; NOW APPLY PAPER THRESHOLD of 0.3
# Convert 1/0 to True/ False
perm_pass = np.array(var_exp_pass) == 1
pc1_pass  = (program_coherence_mb["Variance_expained"] >= 0.3).to_numpy()
validated_regulons = coh_ids[perm_pass & pc1_pass]

print("Regulons passing permutation p<=0.05:", int(perm_pass.sum()))
print("Regulons passing PC1>=0.3 AND p<=0.05:", len(validated_regulons))

# Outputs:
coherence_out = program_coherence_mb.assign(Regulon=coh_ids, Perm_pass=perm_pass.astype(int), PC1_pass=pc1_pass.astype(int))
coherence_out.to_csv(os.path.join(output_dir, f"{val}_regulon_coherence.csv"), index=False)

pd.Series(sorted(validated_regulons)).to_csv(
    os.path.join(output_dir, f"{val}_coexp_validated_regulons.csv"), index=False, header=False)

print(f"Saved {val}_coexp_validated_regulons.csv")

Distinct regulon sizes to simulate: 145 | draws per size: 250 | CPU cores: 22


regulon sizes:   0%|          | 0/145 [00:00<?, ?it/s]

Null distribution done in 1.4 min: (36250, 3)
Regulons passing permutation p<=0.05: 5768
Regulons passing PC1>=0.3 AND p<=0.05: 4021
Saved gse72094_coexp_validated_regulons.csv


## Disease relevance: the paper's rule

The paper's Supplementary Table 1 shows the rule actually applied to its 1,083 disease-relevant regulons. A regulon must be:

1. **Co-expressed in TCGA**: within its coherent members (the patients in whom it is over- or underactive), PC1 explains at least 30% of the variance, and more than random gene sets and random patient sets of the same size (permutation p ≤ 0.05 for both)
2. **Survival-associated in TCGA**: Cox regression on the regulon eigengene, p ≤ 0.05
3. **Co-expressed in the independent cohort**: the same three measures as criterion 1, within the regulon's coherent members in that cohort (the earlier test across all of the cohort's patients is kept for the looser rule)
4. **Survival-associated in the independent cohort**: Cox p ≤ 0.05, with the same direction of effect as in TCGA
5. **Regulator correlated with the regulon** (Methods criterion iii): within the regulon's coherent members in TCGA, Spearman |rho| ≥ 0.3 between the TF's expression and the regulon eigengene (`regulator_eigengene_correlation.csv`)

Criteria 3 and 4 need an independent cohort: a cancer without one uses criteria 1 and 2 and is marked as not externally validated; a cohort without survival data (e.g. UCEC CPTAC) contributes criterion 3 only. Hallmark enrichment is kept as an annotation; it is not part of the rule (in the paper every disease-relevant regulon is survival-associated).

`final_disease_relevant_regulons.csv` holds this set and is read by notebook 07. The earlier rule, (survival OR hallmark) AND co-expression validated, is kept in `final_disease_relevant_regulons_loose.csv`.

In [2]:
import os, json, time, gzip
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# Criterion 1: co-expression of each regulon in TCGA, within its coherent members (patients in whom it is over- or underactive)
# PC1 variance explained, and two permutation p-values (500 permutations each):
#   p_random_genes:    the same coherent patients, random gene sets of the same size
#   p_random_patients: the same genes, random patient sets of the same size
# Runs in parallel over regulons (N_JOBS workers) with a progress bar. Each regulon has its own random seed, so the
# results do not depend on the number of workers or the order in which they finish
# Cached in tcga_regulon_coherence.csv; recomputed when the regulons have changed (delete the file to force it)
from joblib import Parallel, delayed
from tqdm.auto import tqdm

N_PERM = 500
coh_path = os.path.join(output_dir, "tcga_regulon_coherence.csv")
with open(os.path.join(output_dir, "regulons.json")) as f:
    regulons_all = json.load(f)

coherence = pd.read_csv(coh_path, index_col=0) if os.path.exists(coh_path) else None
if coherence is not None and set(coherence.index.astype(int)) != set(int(r) for r in regulons_all):
    print("Cached TCGA coherence does not match the current regulons: recomputing")
    coherence = None


def _coherence_one(X, reg_id, g_idx, m_idx, n_perm, seed):

    '''

    Co-expression test for one regulon. Runs in a separate worker process, hence the import inside.

    Args: X (np.array), shape (n_genes, n_patients). TCGA expression, the same for every regulon
          reg_id (int). Regulon ID
          g_idx (np.array). Row positions of the regulon's genes in X
          m_idx (np.array). Column positions of the regulon's coherent patients in X
          n_perm (int). Number of permutations for each null distribution
          seed (int). Random seed for this regulon

    Returns: dict with regulon, n_genes, n_coherent_patients, pc1_var_explained, p_random_genes, p_random_patients
             (NaN when the regulon has fewer than 3 measured genes or 3 coherent patients)

    '''

    import numpy as np
    row = {"regulon": reg_id, "n_genes": np.nan, "n_coherent_patients": np.nan,
           "pc1_var_explained": np.nan, "p_random_genes": np.nan, "p_random_patients": np.nan}
    if len(g_idx) < 3 or len(m_idx) < 3:
        return row

    def pc1_share(A):
        # Share of variance on PC1 for a batch of gene sets, A of shape (n_sets, n_genes, n_patients)
        # The top eigenvalue is taken from the smaller of the two Gram matrices; the total is the trace
        A = A - A.mean(axis=2, keepdims=True)
        total = np.einsum("ijk,ijk->i", A, A)
        G = A @ A.transpose(0, 2, 1) if A.shape[1] <= A.shape[2] else A.transpose(0, 2, 1) @ A
        return np.linalg.eigvalsh(G)[:, -1] / total

    rng = np.random.default_rng(seed)
    k, m = len(g_idx), len(m_idx)
    observed = pc1_share(X[np.ix_(g_idx, m_idx)][None])[0]

    # Random sets are drawn in batches small enough to keep memory low in every worker
    chunk = max(1, min(n_perm, int(2e6 // (k * m))))
    null_genes, null_pats = [], []
    for s in range(0, n_perm, chunk):
        n = min(chunk, n_perm - s)
        rand_genes = np.array([rng.choice(X.shape[0], size=k, replace=False) for _ in range(n)])
        null_genes.append(pc1_share(X[rand_genes[:, :, None], m_idx[None, None, :]]))
        rand_pats = np.array([rng.choice(X.shape[1], size=m, replace=False) for _ in range(n)])
        null_pats.append(pc1_share(X[g_idx[None, :, None], rand_pats[:, None, :]]))
    null_genes, null_pats = np.concatenate(null_genes), np.concatenate(null_pats)
    row.update({"n_genes": k, "n_coherent_patients": m, "pc1_var_explained": observed,
                "p_random_genes":    (1 + (null_genes >= observed).sum()) / (n_perm + 1),
                "p_random_patients": (1 + (null_pats  >= observed).sum()) / (n_perm + 1)})
    return row


if coherence is not None:
    print(f"Loaded cached TCGA coherence: {len(coherence)} regulons")
else:
    # TCGA expression (genes x patients) with Ensembl row IDs, to match the regulon members
    expr = pd.read_csv(os.path.join(data_dir, f"{prefix}_zscored.csv"), index_col=0)
    idmap = pd.read_csv(idmap_path, sep="\t", header=0)
    gm = idmap[idmap["Source"] == "Gene Name"]
    sym2ens = dict(zip(gm["Name"], gm["Preferred_Name"]))
    expr.index = [sym2ens.get(g, g) for g in expr.index]
    expr = expr[~expr.index.duplicated()]

    over  = pd.read_csv(os.path.join(output_dir, "overExpressedMembers.csv"),  index_col=0)
    under = pd.read_csv(os.path.join(output_dir, "underExpressedMembers.csv"), index_col=0)
    over.index, under.index = over.index.astype(str), under.index.astype(str)

    X_all    = np.ascontiguousarray(expr.values, dtype=np.float64)
    gene_pos = {g: i for i, g in enumerate(expr.index)}
    pat_pos  = {p: i for i, p in enumerate(expr.columns)}

    # One task per regulon, largest first so that the workers finish together
    tasks = []
    for reg_id, genes in regulons_all.items():
        g_idx = np.array([gene_pos[g] for g in genes if g in gene_pos], dtype=int)
        if reg_id in over.index:
            members = over.columns[(over.loc[reg_id] == 1) | (under.loc[reg_id] == 1)]
            m_idx = np.array([pat_pos[p] for p in members if p in pat_pos], dtype=int)
        else:
            m_idx = np.array([], dtype=int)
        tasks.append((len(g_idx) * len(m_idx), int(reg_id), g_idx, m_idx))
    tasks.sort(key=lambda t: -t[0])

    # The matrix goes to the workers as a memory-mapped file. Passed as an ordinary array, joblib re-hashes all of it
    # for every regulon, which makes large cohorts very slow. The file name is unique to each run, because Windows
# cannot overwrite a file that an earlier, interrupted run still has open; it is removed at the end when possible
    import tempfile
    mm_path = os.path.join(tempfile.gettempdir(), f"miner_{prefix}_tcga_expr_{os.getpid()}_{int(time.time())}.npy")
    np.save(mm_path, X_all)
    X_mm = np.load(mm_path, mmap_mode="r")

    start = time.time()
    jobs = (delayed(_coherence_one)(X_mm, reg_id, g_idx, m_idx, N_PERM, 12000 + reg_id) for _, reg_id, g_idx, m_idx in tasks)
    rows = list(tqdm(Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(jobs), total=len(tasks), desc="regulons"))
    del X_mm
    try:
        os.remove(mm_path)
    except OSError:
        pass

    coherence = pd.DataFrame(rows).set_index("regulon").sort_index()
    coherence.to_csv(coh_path)
    print(f"TCGA coherence computed in {(time.time()-start)/60:.1f} min for {len(coherence)} regulons")

coherence.index = coherence.index.astype(int)
ok_var  = coherence["pc1_var_explained"] >= 0.3
ok_gene = coherence["p_random_genes"] <= 0.05
ok_pat  = coherence["p_random_patients"] <= 0.05
coherent_tcga = set(coherence.index[ok_var & ok_gene & ok_pat])
print(f"PC1 >= 0.3: {ok_var.sum()} | p (random genes) <= 0.05: {ok_gene.sum()} | p (random patients) <= 0.05: {ok_pat.sum()}")
print(f"Co-expressed in TCGA (all three): {len(coherent_tcga)} of {len(coherence)}")

regulons:   0%|          | 0/7447 [00:00<?, ?it/s]

TCGA coherence computed in 4.8 min for 7447 regulons
PC1 >= 0.3: 7439 | p (random genes) <= 0.05: 7417 | p (random patients) <= 0.05: 7408
Co-expressed in TCGA (all three): 7393 of 7447


In [3]:
import os, json, time, gzip
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# Criterion 3: co-expression of each regulon in the independent cohort, within its coherent members there, as in the paper
# (its Supplementary Table 1 computes the cohort columns "same as above", i.e. within coherent members)
# Coherent members = the cohort's patients in whom the regulon is over- or underactive, scored with the same miner method as
# for TCGA (generateRegulonActivity). That activity is cached in <cohort>_regulon_activity.pkl, the file notebook 05 also uses
# Then the same three measures as criterion 1, using the test function defined in the TCGA cell above
# Gravendeel: the paper uses grade II to IV gliomas only, so grade I samples are left out
# Cached in <cohort>_regulon_coherence_members.csv; recomputed when the regulons have changed
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from miner.miner import generateRegulonActivity
assert "_coherence_one" in globals(), "Run the TCGA co-expression cell (criterion 1) first"

HAS_VALIDATION = VAL_NAME is not None
coherent_ext = None
ext_expr_path = os.path.join(output_dir, f"{val}_zscored.pkl")

if HAS_VALIDATION and os.path.exists(ext_expr_path):
    with open(os.path.join(output_dir, "regulons.json")) as f:
        regulons_all = json.load(f)
    ext_coh_path = os.path.join(output_dir, f"{val}_regulon_coherence_members.csv")
    ext_coh = pd.read_csv(ext_coh_path, index_col=0) if os.path.exists(ext_coh_path) else None
    if ext_coh is not None and set(ext_coh.index.astype(int)) != set(int(r) for r in regulons_all):
        print(f"Cached {VAL_NAME} coherence does not match the current regulons: recomputing")
        ext_coh = None

    if ext_coh is not None:
        print(f"Loaded cached {VAL_NAME} coherence within coherent members: {len(ext_coh)} regulons")
    else:
        ext_expr = pd.read_pickle(ext_expr_path)
        ext_expr.columns = ext_expr.columns.astype(str)

        # Regulon genes as symbols, restricted to genes measured in the cohort (regulons with < 2 such genes cannot be scored)
        idmap = pd.read_csv(idmap_path, sep="\t", header=0)
        gm = idmap[idmap["Source"] == "Gene Name"]
        ensembl_to_symbol = dict(zip(gm["Preferred_Name"], gm["Name"]))
        ext_genes = set(ext_expr.index)
        regulon_modules_sym = {}
        for reg_id, genes in regulons_all.items():
            syms = [s for s in dict.fromkeys(ensembl_to_symbol.get(g, g) for g in genes) if s in ext_genes]
            if len(syms) >= 2:
                regulon_modules_sym[int(reg_id)] = syms

        # Regulon activity in the cohort (1 / 0 / -1 per regulon and sample); reuses notebook 05's cache when it matches
        act_path = os.path.join(output_dir, f"{val}_regulon_activity.pkl")
        act = pd.read_pickle(act_path) if os.path.exists(act_path) else None
        if act is not None:
            act.columns = act.columns.astype(str)
            if set(act.index.astype(int)) != set(regulon_modules_sym) or not set(ext_expr.columns) <= set(act.columns):
                print(f"Cached {VAL_NAME} regulon activity does not match the current regulons or samples: recomputing")
                act = None
        if act is None:
            start = time.time()
            act = generateRegulonActivity(regulon_modules_sym, ext_expr)
            act.index = act.index.astype(int)
            act.to_pickle(act_path)
            print(f"{VAL_NAME} regulon activity: {act.shape} in {(time.time()-start)/60:.1f} min")
        act.index = act.index.astype(int)
        act.columns = act.columns.astype(str)

        # Safety check: miner.background_df silently returns all zeros under pandas >= 3 (copy-on-write)
        assert (act != 0).values.mean() > 0.01, f"regulon activity ~all zero (pandas {pd.__version__}); use pandas<3"

        # Samples used: all, except Gravendeel, where grade I gliomas are left out
        samples = list(ext_expr.columns)
        if VAL_NAME == "GRAVENDEEL":
            histology, title = {}, None
            with gzip.open(os.path.join(val_dir, "GSE16011_family.soft.gz"), "rt", errors="ignore") as f:
                for line in f:
                    if line.startswith("!Sample_title"):
                        title = line.split("=", 1)[1].strip()
                    elif line.startswith("!Sample_characteristics_ch1 = histology:"):
                        histology[title.split()[-1]] = line.split("histology:", 1)[1].strip()
            samples = [p for p in samples if histology.get(p, "").endswith(("(grade II)", "(grade III)", "(grade IV)"))]

        # Genes with missing values in more than 10% of samples are dropped; remaining gaps are set to 0 (the mean of a z-score)
        E = ext_expr.loc[:, samples]
        E = E[E.isna().mean(axis=1) <= 0.1].fillna(0)
        X_ext    = np.ascontiguousarray(E.values, dtype=np.float64)
        gene_pos = {g: i for i, g in enumerate(E.index)}
        pat_pos  = {p: i for i, p in enumerate(E.columns)}
        print(f"{VAL_NAME}: {X_ext.shape[1]} samples, {X_ext.shape[0]} genes")

        tasks = []
        for reg_id in regulons_all:
            rid = int(reg_id)
            syms = regulon_modules_sym.get(rid, [])
            g_idx = np.array([gene_pos[s] for s in syms if s in gene_pos], dtype=int)
            if rid in act.index:
                row = act.loc[rid, samples]
                m_idx = np.array([pat_pos[p] for p in row.index[row != 0]], dtype=int)
            else:
                m_idx = np.array([], dtype=int)
            tasks.append((len(g_idx) * len(m_idx), rid, g_idx, m_idx))
        tasks.sort(key=lambda t: -t[0])

        # The matrix goes to the workers as a memory-mapped file. Passed as an ordinary array, joblib re-hashes all of it
        # for every regulon, which makes large cohorts very slow. The file name is unique to each run, because Windows
# cannot overwrite a file that an earlier, interrupted run still has open; it is removed at the end when possible
        import tempfile
        mm_path = os.path.join(tempfile.gettempdir(), f"miner_{prefix}_{val}_expr_{os.getpid()}_{int(time.time())}.npy")
        np.save(mm_path, X_ext)
        X_mm = np.load(mm_path, mmap_mode="r")

        start = time.time()
        jobs = (delayed(_coherence_one)(X_mm, rid, g_idx, m_idx, N_PERM, 22000 + rid) for _, rid, g_idx, m_idx in tasks)
        rows = list(tqdm(Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(jobs), total=len(tasks), desc=f"regulons in {VAL_NAME}"))
        del X_mm
        try:
            os.remove(mm_path)
        except OSError:
            pass
        ext_coh = pd.DataFrame(rows).set_index("regulon").sort_index()
        ext_coh.to_csv(ext_coh_path)
        print(f"{VAL_NAME} coherence computed in {(time.time()-start)/60:.1f} min for {len(ext_coh)} regulons")

    ext_coh.index = ext_coh.index.astype(int)
    ok_var  = ext_coh["pc1_var_explained"] >= 0.3
    ok_gene = ext_coh["p_random_genes"] <= 0.05
    ok_pat  = ext_coh["p_random_patients"] <= 0.05
    coherent_ext = set(ext_coh.index[ok_var & ok_gene & ok_pat])
    print(f"{VAL_NAME}, within coherent members | PC1 >= 0.3: {ok_var.sum()} | p (random genes) <= 0.05: {ok_gene.sum()} | p (random patients) <= 0.05: {ok_pat.sum()}")
    print(f"Co-expressed in {VAL_NAME} (all three): {len(coherent_ext)} of {len(ext_coh)}")
elif HAS_VALIDATION:
    print(f"{VAL_NAME} expression not found ({ext_expr_path}): run the validation cells above first")
else:
    print("No independent cohort for this cancer: criterion 3 cannot be applied")

GSE72094: 442 samples, 21283 genes


regulons in GSE72094:   0%|          | 0/7447 [00:00<?, ?it/s]

GSE72094 coherence computed in 3.5 min for 7447 regulons
GSE72094, within coherent members | PC1 >= 0.3: 7357 | p (random genes) <= 0.05: 7424 | p (random patients) <= 0.05: 7411
Co-expressed in GSE72094 (all three): 7316 of 7447


In [4]:
import os, json, time, gzip
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# Criterion 4: survival association replicated in the independent cohort (Cox regression on each regulon's eigengene)
# Needs the cohort's z-scored expression and survival table, both written by the validation cells above
# Gravendeel: the paper uses grade II to IV gliomas only, so grade I samples are left out
# Cached in <cohort>_regulon_cox_results.csv; recomputed when the regulons have changed
HAS_VALIDATION = VAL_NAME is not None
surv_ext = None
ext_clin_path = os.path.join(output_dir, f"{val}_clinical.csv")
ext_expr_path = os.path.join(output_dir, f"{val}_zscored.pkl")
ext_has_survival = (HAS_VALIDATION and os.path.exists(ext_clin_path) and os.path.exists(ext_expr_path)
                    and {"OS_MONTHS", "OS_STATUS"} <= set(pd.read_csv(ext_clin_path, index_col=0, nrows=0).columns))

if ext_has_survival:
    with open(os.path.join(output_dir, "regulons.json")) as f:
        regulons_all = json.load(f)
    ext_cox_path = os.path.join(output_dir, f"{val}_regulon_cox_results.csv")
    ext_cox = pd.read_csv(ext_cox_path, index_col=0) if os.path.exists(ext_cox_path) else None
    if ext_cox is not None and set(ext_cox.index.astype(int)) != set(int(r) for r in regulons_all):
        print(f"Cached {VAL_NAME} Cox results do not match the current regulons: recomputing")
        ext_cox = None

    if ext_cox is not None:
        print(f"Loaded cached {VAL_NAME} Cox results: {len(ext_cox)} regulons")
    else:
        ext_expr = pd.read_pickle(ext_expr_path)
        ext_expr.columns = ext_expr.columns.astype(str)
        ext_surv = pd.read_csv(ext_clin_path, index_col=0)
        ext_surv.index = ext_surv.index.astype(str)
        ext_ids = [p for p in ext_expr.columns if p in ext_surv.index]

        if VAL_NAME == "GRAVENDEEL":
            histology, title = {}, None
            with gzip.open(os.path.join(val_dir, "GSE16011_family.soft.gz"), "rt", errors="ignore") as f:
                for line in f:
                    if line.startswith("!Sample_title"):
                        title = line.split("=", 1)[1].strip()
                    elif line.startswith("!Sample_characteristics_ch1 = histology:"):
                        histology[title.split()[-1]] = line.split("histology:", 1)[1].strip()
            ext_ids = [p for p in ext_ids if histology.get(p, "").endswith(("(grade II)", "(grade III)", "(grade IV)"))]

        ext_surv = ext_surv.loc[ext_ids, ["OS_MONTHS", "OS_STATUS"]].dropna()
        ext_ids = list(ext_surv.index)
        print(f"{VAL_NAME} patients used: {len(ext_ids)} | deaths: {int(ext_surv['OS_STATUS'].sum())}")

        idmap = pd.read_csv(idmap_path, sep="\t", header=0)
        gm = idmap[idmap["Source"] == "Gene Name"]
        ensembl_to_symbol = dict(zip(gm["Preferred_Name"], gm["Name"]))
        E = ext_expr.loc[:, ext_ids].dropna()
        ext_gene_set = set(E.index)

        def eigengene(M):

            '''

            First principal component of a regulon's genes across patients, oriented so that it rises with mean expression.

            Args: M (np.array), shape (n_genes, n_patients). Z-scored expression of the regulon's genes

            Returns: pc (np.array), shape (n_patients,). Eigengene, scaled to unit standard deviation

            '''

            M = M - M.mean(axis=1, keepdims=True)
            pc = np.linalg.svd(M, full_matrices=False)[2][0]
            if np.corrcoef(pc, M.mean(axis=0))[0, 1] < 0:
                pc = -pc
            return pc / pc.std()

        from tqdm.auto import tqdm
        start, rows = time.time(), []
        for reg_id, genes in tqdm(regulons_all.items(), total=len(regulons_all), desc=f"Cox in {VAL_NAME}"):
            syms = [s for s in dict.fromkeys(ensembl_to_symbol.get(g, g) for g in genes) if s in ext_gene_set]
            row = {"regulon": int(reg_id), "HR": np.nan, "p_value": np.nan}
            if len(syms) >= 3:
                df_cox = ext_surv.copy()
                df_cox["eigengene"] = eigengene(E.loc[syms].values)
                try:
                    cph = CoxPHFitter().fit(df_cox, duration_col="OS_MONTHS", event_col="OS_STATUS")
                    row.update({"HR": cph.hazard_ratios_["eigengene"], "p_value": cph.summary.loc["eigengene", "p"]})
                except Exception:
                    pass
            rows.append(row)
        ext_cox = pd.DataFrame(rows).set_index("regulon").sort_index()
        ext_cox.to_csv(ext_cox_path)
        print(f"Cox fitted for {len(ext_cox)} regulons in {(time.time()-start)/60:.1f} min")

    # Survival-associated in the cohort, with the same direction of effect as in TCGA
    ext_cox.index = ext_cox.index.astype(int)
    tcga_cox = pd.read_csv(os.path.join(output_dir, "regulon_cox_results.csv"), index_col=0)
    tcga_cox.index = tcga_cox.index.astype(int)
    both = tcga_cox[["HR"]].join(ext_cox[["HR", "p_value"]], rsuffix="_ext").dropna()
    surv_ext = set(both.index[(both["p_value"] <= 0.05) & ((both["HR"] > 1) == (both["HR_ext"] > 1))])
    print(f"Survival-associated in {VAL_NAME} (p <= 0.05, same direction as TCGA): {len(surv_ext)}")
elif HAS_VALIDATION:
    print(f"{VAL_NAME} has no survival data: criterion 4 cannot be applied")
else:
    print("No independent cohort for this cancer: criterion 4 cannot be applied")

GSE72094 patients used: 398 | deaths: 113


Cox in GSE72094:   0%|          | 0/7447 [00:00<?, ?it/s]

Cox fitted for 7447 regulons in 13.8 min
Survival-associated in GSE72094 (p <= 0.05, same direction as TCGA): 2967


In [5]:
import os, json, time, gzip
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# Methods criterion (iii): the regulon's TF is correlated with the regulon eigengene within the regulon's coherent members
# (the TCGA patients in whom the regulon is over- or underactive), Spearman |rho| >= 0.3. The paper states only the
# correlation, no p-value. Regulons whose TF has no expression data, or with fewer than MIN_MEMBERS coherent patients, fail.
from scipy.stats import spearmanr

MIN_REG_RHO = 0.3
MIN_MEMBERS = 5

# TCGA expression (gene symbols x patients), regulon eigengenes and coherent members from 02
expr_path = next(p for p in (os.path.join(globals().get("data_dir", base_dir), f"{prefix}_zscored.csv"),
                             os.path.join(base_dir, f"{prefix}_zscored.csv")) if os.path.exists(p))
expr_sym = pd.read_csv(expr_path, index_col=0)
eig      = pd.read_csv(os.path.join(output_dir, "eigengenes.csv"), index_col=0)
eig.index = eig.index.astype(str)
over_m   = pd.read_csv(os.path.join(output_dir, "overExpressedMembers.csv"),  index_col=0)
under_m  = pd.read_csv(os.path.join(output_dir, "underExpressedMembers.csv"), index_col=0)
over_m.index, under_m.index = over_m.index.astype(str), under_m.index.astype(str)

# TF of each regulon, as a gene symbol
idmap = pd.read_csv(idmap_path, sep="\t", header=0)
gm = idmap[idmap["Source"] == "Gene Name"]
ens2sym = dict(zip(gm["Preferred_Name"], gm["Name"]))
reg_tf = (pd.read_csv(os.path.join(output_dir, "regulonDf.csv"), index_col=0)
          .drop_duplicates("Regulon_ID").set_index("Regulon_ID")["Regulator"])
reg_tf.index = reg_tf.index.astype(str)

rows = []
for rid, tf_ens in reg_tf.items():
    tf_sym = ens2sym.get(tf_ens, tf_ens)
    row = {"regulon": int(rid), "TF": tf_sym, "n_coherent_patients": 0, "rho": np.nan}
    if rid in over_m.index and rid in eig.index and tf_sym in expr_sym.index:
        members = over_m.columns[(over_m.loc[rid] == 1) | (under_m.loc[rid] == 1)]
        members = [p for p in members if p in eig.columns and p in expr_sym.columns]
        row["n_coherent_patients"] = len(members)
        if len(members) >= MIN_MEMBERS:
            row["rho"] = spearmanr(expr_sym.loc[tf_sym, members].values, eig.loc[rid, members].values)[0]
    rows.append(row)

regulator_corr = pd.DataFrame(rows).set_index("regulon").sort_index()
regulator_corr["pass"] = regulator_corr["rho"].abs() >= MIN_REG_RHO
regulator_corr.to_csv(os.path.join(output_dir, "regulator_eigengene_correlation.csv"))
mech_ok = set(regulator_corr.index[regulator_corr["pass"]])

print(f"Regulons: {len(regulator_corr)} | TF without expression data: {int((~regulator_corr['TF'].isin(expr_sym.index)).sum())} "
      f"| fewer than {MIN_MEMBERS} coherent patients: {int((regulator_corr['n_coherent_patients'] < MIN_MEMBERS).sum())}")
print(f"|rho| >= {MIN_REG_RHO} within coherent members (criterion iii): {len(mech_ok)} of {len(regulator_corr)} "
      f"| median |rho|: {regulator_corr['rho'].abs().median():.2f}")

Regulons: 7447 | TF without expression data: 0 | fewer than 5 coherent patients: 0
|rho| >= 0.3 within coherent members (criterion iii): 4607 of 7447 | median |rho|: 0.37


In [6]:
# Final disease-relevant regulons under the paper's rule (see the markdown cell above)
# Also saves the earlier, looser rule for reference: (survival OR hallmark) AND co-expression validated
HAS_VALIDATION = VAL_NAME is not None
survival_set = set(pd.read_csv(os.path.join(output_dir, "disease_relevant_regulons_survival_only.csv"), header=None)[0].astype(int))
union_set    = set(pd.read_csv(os.path.join(output_dir, "disease_relevant_regulons_survival_or_hallmark.csv"), header=None)[0].astype(int))
print(f"Survival-associated in TCGA (criterion 2): {len(survival_set)} | survival OR hallmark: {len(union_set)}")

final_regulons = survival_set & coherent_tcga
print(f"  and co-expressed in TCGA (criterion 1): {len(final_regulons)}")

# Methods criterion (iii): regulator correlated with the eigengene within coherent members (cell above)
final_regulons &= mech_ok
print(f"  and regulator |rho| >= {MIN_REG_RHO} within coherent members (criterion 5 / Methods iii): {len(final_regulons)}")

if HAS_VALIDATION:
    # Criterion 3 uses the test within coherent members (cell above); the earlier all-patient test feeds the looser rule only
    old_coexp_set = set(pd.read_csv(os.path.join(output_dir, f"{val}_coexp_validated_regulons.csv"), header=None)[0].astype(int))
    loose_regulons = union_set & old_coexp_set
    coexp_set = coherent_ext if coherent_ext is not None else old_coexp_set
    if coherent_ext is None:
        print("  (criterion 3 within coherent members not available: using the all-patient test)")
    final_regulons &= coexp_set
    print(f"  and co-expressed in {VAL_NAME} (criterion 3): {len(final_regulons)}")
    if surv_ext is not None:
        final_regulons &= surv_ext
        print(f"  and survival-associated in {VAL_NAME}, same direction (criterion 4): {len(final_regulons)}")
        status = f"externally validated in {VAL_NAME}: co-expression and survival (paper's rule, criteria 1-5)"
    else:
        status = f"partly validated in {VAL_NAME}: co-expression only, the cohort has no survival data (criteria 1-3 and 5)"
else:
    loose_regulons = union_set
    status = "NOT externally validated: no independent cohort (criteria 1, 2 and 5 only)"

print(f"Final disease-relevant regulons: {len(final_regulons)} | status: {status}")
print(f"Earlier, looser rule: {len(loose_regulons)}")

# Written WITHOUT a header; always read back with header=None
pd.Series(sorted(final_regulons)).to_csv(os.path.join(output_dir, "final_disease_relevant_regulons.csv"), index=False, header=False)
pd.Series(sorted(loose_regulons)).to_csv(os.path.join(output_dir, "final_disease_relevant_regulons_loose.csv"), index=False, header=False)

# Records how this cancer's results were checked in an independent cohort
with open(os.path.join(output_dir, "validation_status.txt"), "w") as f:
    f.write(status + "\n")

Survival-associated in TCGA (criterion 2): 2919 | survival OR hallmark: 3266
  and co-expressed in TCGA (criterion 1): 2902
  and regulator |rho| >= 0.3 within coherent members (criterion 5 / Methods iii): 1868
  and co-expressed in GSE72094 (criterion 3): 1842
  and survival-associated in GSE72094, same direction (criterion 4): 1068
Final disease-relevant regulons: 1068 | status: externally validated in GSE72094: co-expression and survival (paper's rule, criteria 1-5)
Earlier, looser rule: 2026
